# Module 12: Agent Protocols - 02: Two local agents, talking A2A

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 introduced the three A2A objects in isolation. This notebook
builds **two complete, independent agents** - a `TravelerAgent` and a
`SkyBookerAgent` - and runs a real exchange between them: discovery, a
delegated task, a genuine mid-task clarifying question, and completion.

Both agents run **in this notebook's process**, but they are written as if
they were not - every exchange goes through message objects shaped exactly
like the ones a real HTTP-based A2A server and client would send. That is
the honest scope of "two local agents": we implement the protocol's message
shapes ourselves rather than depending on a hosted A2A deployment or the
`a2a-sdk` package, exactly as notebook 01 said we would.

### What you will learn

1. Implementing an agent as a **request handler** driven entirely by
   incoming Task/Message objects, with no shared state with its caller.
2. A real **`input-required`** round trip - the protocol earning its keep.
3. What "server" and "client" mean in A2A (both sides can act as either).

No LLM calls, no API key.

### Setup: reuse the message shapes from notebook 01


In [ ]:
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional


@dataclass
class Skill:
    id: str
    name: str
    description: str
    examples: list = field(default_factory=list)


@dataclass
class AgentCard:
    name: str
    description: str
    url: str
    version: str
    skills: list = field(default_factory=list)


@dataclass
class Part:
    kind: str
    text: Optional[str] = None
    data: Optional[dict] = None


@dataclass
class Message:
    role: str
    parts: list
    task_id: str
    message_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])


@dataclass
class Task:
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    status: str = "submitted"
    history: list = field(default_factory=list)
    artifacts: list = field(default_factory=list)


def show(msg: Message, sender: str):
    text = next((p.text for p in msg.parts if p.kind == "text"), None)
    data = next((p.data for p in msg.parts if p.kind == "data"), None)
    line = f"[{sender:>10} -> task {msg.task_id}] {text or ''}"
    if data:
        line += f"  data={data}"
    print(line)


print("shared A2A message shapes ready")


### 1. An agent, implemented as a Task handler

A real A2A server exposes an HTTP endpoint that receives Task/Message
payloads and returns updated ones. We simulate that boundary with a plain
Python class whose only public method, `handle(task, message)`, plays exactly
that role: it receives a message, does some work, and returns its response -
with **no access to anything about the caller except what arrived in the
message**. That restriction is not a simplification for the demo; it is the
actual constraint a real A2A server operates under, and it is what makes this
"two agents that don't share a runtime" rather than "two Python objects
calling each other's methods directly."

### SkyBookerAgent: the "server" side


In [ ]:
class SkyBookerAgent:
    """A flight-booking agent. Knows nothing about its caller except what
    arrives in each Message -- no shared Python objects, no shared state."""

    def __init__(self):
        self.card = AgentCard(
            name="SkyBooker",
            description="Books and manages flight reservations.",
            url="local://skybooker",
            version="1.0.0",
            skills=[
                Skill(id="book-flight", name="Book Flight",
                     description="Reserve a specific flight for a named passenger.",
                     examples=["Book flight SK123 for Ada Lovelace"]),
            ],
        )
        self._tasks: dict = {}          # this agent's OWN view of task state
        self._inventory = {
            "SK123": {"seats_left": 3, "route": "JFK-LHR", "fare_classes": ["economy", "business"]},
        }

    def handle(self, task: Task, message: Message) -> tuple[Task, Message]:
        """The one entry point a caller ever touches. Everything the agent
        needs to make a decision must be IN `task` and `message` -- that is
        the whole point of the protocol boundary."""
        text = next((p.text for p in message.parts if p.kind == "text"), "")
        data = next((p.data for p in message.parts if p.kind == "data"), {}) or {}

        # Track OUR OWN copy of the task -- an A2A server does not trust a
        # caller's local Task object, it maintains authoritative state.
        local = self._tasks.setdefault(task.id, {"stage": "new"})

        if local["stage"] == "new":
            flight = data.get("flight_id") or self._extract_flight_id(text)
            if flight is None or flight not in self._inventory:
                local["stage"] = "failed"
                task.status = "failed"
                reply = Message(role="agent", task_id=task.id,
                                parts=[Part(kind="text", text=f"Unknown flight in request: {text!r}")])
                return task, reply

            # THE INTERESTING CASE: the request is understandable but
            # incomplete. Rather than guessing, the agent asks -- exactly
            # the multi-turn negotiation a plain function call cannot do.
            if "fare_class" not in data:
                local.update(stage="awaiting_fare_class", flight=flight)
                task.status = "input-required"
                reply = Message(role="agent", task_id=task.id,
                                parts=[Part(kind="text",
                                          text=f"Flight {flight} found ({self._inventory[flight]['seats_left']} seats left). "
                                               f"Which fare class: {self._inventory[flight]['fare_classes']}?")])
                return task, reply

            return self._finalize(task, local, data["fare_class"])

        if local["stage"] == "awaiting_fare_class":
            fare_class = data.get("fare_class")
            if fare_class not in self._inventory[local["flight"]]["fare_classes"]:
                reply = Message(role="agent", task_id=task.id,
                                parts=[Part(kind="text", text=f"'{fare_class}' is not a valid fare class.")])
                return task, reply   # status stays input-required
            return self._finalize(task, local, fare_class)

        # stage already terminal
        reply = Message(role="agent", task_id=task.id,
                        parts=[Part(kind="text", text=f"Task {task.id} is already {task.status}.")])
        return task, reply

    def _finalize(self, task, local, fare_class):
        flight = local["flight"]
        self._inventory[flight]["seats_left"] -= 1
        local["stage"] = "completed"
        task.status = "completed"
        artifact = {"confirmation": f"CONF-{task.id.upper()}", "flight": flight, "fare_class": fare_class}
        task.artifacts.append(artifact)
        reply = Message(role="agent", task_id=task.id,
                        parts=[Part(kind="text", text=f"Booked {flight} ({fare_class}). Confirmation {artifact['confirmation']}."),
                              Part(kind="data", data=artifact)])
        return task, reply

    @staticmethod
    def _extract_flight_id(text: str) -> Optional[str]:
        for word in text.split():
            if word.upper().startswith("SK") and word[2:].isdigit():
                return word.upper()
        return None


skybooker = SkyBookerAgent()
print("SkyBookerAgent ready. Skills:", [s.id for s in skybooker.card.skills])


### 2. The caller: discover, delegate, and handle `input-required`

The caller side is a `TravelerAgent`. Its `run()` loop is the general shape
any A2A client needs: send a message, look at the returned status, and if
that status is `input-required`, decide how to answer and send again - until
the task reaches a terminal status.

### TravelerAgent: the "client" side


In [ ]:
class TravelerAgent:
    """Delegates a booking to another agent it has never called before,
    knowing only that agent's published Agent Card."""

    def __init__(self, preferred_fare: str = "economy"):
        self.preferred_fare = preferred_fare

    def discover(self, card: AgentCard, need: str) -> Optional[Skill]:
        need_words = set(need.lower().split())
        for skill in card.skills:
            haystack = set(f"{skill.name} {skill.description}".lower().split())
            if need_words & haystack:
                return skill
        return None

    def run(self, remote: SkyBookerAgent, request_text: str, max_turns: int = 5) -> Task:
        skill = self.discover(remote.card, request_text)
        print(f"[traveler] discovered skill: {skill.id if skill else None} on '{remote.card.name}'")
        if skill is None:
            raise RuntimeError("no matching skill on remote agent")

        task = Task()
        message = Message(role="user", task_id=task.id,
                          parts=[Part(kind="text", text=request_text)])
        show(message, "traveler")

        for turn in range(max_turns):
            task, reply = remote.handle(task, message)
            show(reply, remote.card.name)

            if task.status in ("completed", "failed"):
                return task

            if task.status == "input-required":
                # A REAL negotiation: the traveler reads the agent's question
                # and answers it -- this could route to a human, an LLM
                # call, or (as here) simple domain logic.
                answer = self._answer(reply)
                message = Message(role="user", task_id=task.id,
                                  parts=[Part(kind="text", text=answer["text"]),
                                        Part(kind="data", data=answer["data"])])
                show(message, "traveler")
                continue

        raise RuntimeError(f"task did not terminate within {max_turns} turns")

    def _answer(self, question: Message) -> dict:
        text = next((p.text for p in question.parts if p.kind == "text"), "")
        if "fare class" in text.lower():
            return {"text": f"{self.preferred_fare}, please.",
                   "data": {"fare_class": self.preferred_fare}}
        return {"text": "I don't understand the question.", "data": {}}


traveler = TravelerAgent(preferred_fare="business")
print("TravelerAgent ready\n")

final_task = traveler.run(skybooker, "Book flight SK123 for Ada Lovelace")
print(f"\nFinal status: {final_task.status}")
print(f"Artifacts   : {final_task.artifacts}")


### Reading the exchange

Count the turns. `SkyBooker` did not have `fare_class` in the first message,
so it replied with `status=input-required` and a question - a real fork in
the protocol, not a hard-coded script. `TravelerAgent`'s `run()` loop read
that status, produced an answer (from its own preference, no LLM involved
here), and sent a second message on the **same** `task_id`. Only then did
`SkyBooker` finalise and return `status=completed` with an artifact.

Neither class imported the other beyond exchanging these plain dataclasses.
`SkyBookerAgent` never saw `TravelerAgent`'s Python object; `TravelerAgent`
never saw `SkyBookerAgent`'s inventory dict. Everything crossed the boundary
as a `Task`/`Message` pair - exactly the discipline an HTTP call enforces for
free, and exactly what a same-process integration (a LangGraph subgraph, a
CrewAI crew) never has to think about because it never has that boundary.

### 3. What happens when discovery fails or the request is malformed

A production integration has to handle the unhappy paths too: an unknown
skill, or a request that names a flight the remote agent has never heard of.

### The failure paths


In [ ]:
print("=== Case: no matching skill ===")
try:
    traveler.run(skybooker, "Please water my houseplants while I am away")
except RuntimeError as e:
    print(f"  caller refused before ever contacting the remote agent: {e}")

print("\n=== Case: skill matches, but the flight does not exist ===")
bad_task = traveler.run(skybooker, "Book flight SK999 for Grace Hopper")
print(f"  final status: {bad_task.status}")


Notice the difference between these two failures. The **first** never reached
`SkyBooker` at all - `discover()` found no matching skill and the caller
refused locally, exactly as it should: there is no reason to make a network
call (or, here, a Python call across the simulated boundary) for a request
the remote agent's own Agent Card says it cannot handle. The **second**
genuinely delegates and gets a clean `status=failed` back, because the
mismatch (an unknown flight number) can only be discovered by the agent that
owns the inventory.

Getting this distinction right matters in a real deployment: failing fast
locally on a capability mismatch is cheap; every case that legitimately needs
the remote agent's own knowledge has to make the round trip.

### Key takeaways

- An A2A agent is a **request handler** that sees only what arrives in a
  `Task`/`Message` pair - no shared Python objects, no shared state, mirroring
  the constraint a real HTTP boundary imposes.
- The `input-required` status is not decoration: it is what let `SkyBooker`
  ask a genuine clarifying question and `TravelerAgent` answer it, all on the
  same `task_id`, across multiple real turns.
- **Discovery failing locally is a feature.** Checking the Agent Card before
  ever contacting the remote agent avoids a wasted round trip for a request
  it was never going to be able to serve.
- Even entirely in-process, structuring the exchange around explicit message
  objects (rather than direct method calls) is what makes this a faithful
  simulation of two agents that do not share a codebase.

**Next:** `03_a2a_vs_mcp.ipynb` - placing A2A next to MCP directly, with a
worked example of when you would reach for one, the other, or both together.